# Exam January 2024 — Solved Notebook with Bangla/Banglish Explanation

Apu, ei notebook-ta **directly Jupyter/Colab-e run korar moto kore** banano.

Important note:
- **Problem 1 and Problem 2 fully solved.**
- **Problem 3 Markov chain-er exact answer diagram chara possible na**, because original notebook-e images chilo `pictures/MarkovA.png`, `MarkovB.png`, etc. Ei images file upload hoy nai. Tai Problem 3-er jonno ami helper code + method diye diyechi. Tumi diagram-er screenshot dile exact matrix and answers fill kore dewa jabe.

In [ ]:
# Common imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Exam notebook-e Utils.timeout thakte pare. Na thakle fallback timeout use korbo.
try:
    from Utils import timeout
except Exception:
    def timeout(func):
        return func

# Problem 1 — Sampling and Monte Carlo Integration

Given CDF:

$$
F(x)=\frac{e^{x^2}-1}{e-1}, \quad 0<x<1
$$

CDF theke inverse ber korbo.

Let:

$$
U=F(x)=\frac{e^{x^2}-1}{e-1}
$$

Then:

$$
U(e-1)=e^{x^2}-1
$$

$$
1+U(e-1)=e^{x^2}
$$

log nile:

$$
x^2=\log(1+U(e-1))
$$

So:

$$
x=\sqrt{\log(1+U(e-1))}
$$

Ei formula use kore sample generate korbo.

In [ ]:
# Problem 1, Part 1

@timeout
def problem1_inversion(n_samples=1):
    """
    Generate samples from F(x) = (e^(x^2)-1)/(e-1), 0<x<1
    using inverse transform sampling.
    """
    U = np.random.uniform(0, 1, n_samples)
    X = np.sqrt(np.log(1 + U * (np.e - 1)))
    return X

## Problem 1, Part 2 — Generate 100000 samples and plot histogram

Density holo derivative of CDF:

$$
f(x)=F'(x)=\frac{2xe^{x^2}}{e-1}
$$

Histogram-er sathe true density plot korle bujha jabe sampling thik hocche kina.

In [ ]:
# Problem 1, Part 2

problem1_samples = problem1_inversion(100000)

x_grid = np.linspace(0, 1, 500)
true_density = (2 * x_grid * np.exp(x_grid**2)) / (np.e - 1)

plt.figure(figsize=(7, 4))
plt.hist(problem1_samples, bins=50, density=True, alpha=0.5, label="Sample histogram")
plt.plot(x_grid, true_density, label="True density")
plt.xlabel("x")
plt.ylabel("Density")
plt.title("Problem 1: Samples vs True Density")
plt.legend()
plt.show()

## Problem 1, Part 3 — Monte Carlo Integral

Integral:

$$
\int_0^1 \sin(x)\frac{2xe^{x^2}}{e-1}dx
$$

Here:

$$
\frac{2xe^{x^2}}{e-1}=f(x)
$$

So integral actually expectation:

$$
E[\sin(X)]
$$

Monte Carlo approximation:

$$
E[\sin(X)] \approx \frac{1}{n}\sum_{i=1}^n \sin(X_i)
$$

In [ ]:
# Problem 1, Part 3

problem1_integral = float(np.mean(np.sin(problem1_samples)))
problem1_integral

## Problem 1, Part 4 — 95% Hoeffding Confidence Interval

Hoeffding interval formula:

$$
\epsilon=(b-a)\sqrt{\frac{\log(2/\alpha)}{2n}}
$$

Here we average \(\sin(X)\). Since \(0<X<1\):

$$
0<\sin(X)<\sin(1)
$$

So:

$$
a=0, \quad b=\sin(1)
$$

95% confidence means:

$$
\alpha=0.05
$$

In [ ]:
# Problem 1, Part 4

n = len(problem1_samples)
alpha = 0.05

a = 0
b = np.sin(1)

epsilon = (b - a) * np.sqrt(np.log(2 / alpha) / (2 * n))

problem1_interval = (
    float(problem1_integral - epsilon),
    float(problem1_integral + epsilon)
)

problem1_interval

## Problem 1, Part 5 — Tricky Distribution

Given CDF:

$$
F(x)=20xe^{20-1/x}, \quad 0<x<\frac{1}{20}
$$

Ei distribution direct sample kora tricky. Shortcut:

Let:

$$
Z=\frac{1}{X}
$$

Then:

$$
Z>20
$$

We choose proposal:

$$
Z=20+Exp(1)
$$

Then:

$$
X=\frac{1}{Z}
$$

Acceptance ratio becomes:

$$
\frac{20(1/Z+1/Z^2)}{1.05}
$$

Why 1.05? Because maximum of \(20(1/Z+1/Z^2)\) for \(Z\ge 20\) occurs at \(Z=20\):

$$
20\left(\frac{1}{20}+\frac{1}{400}\right)=1.05
$$

In [ ]:
# Problem 1, Part 5

def problem1_inversion_2(n_samples=1):
    """
    Generate samples from F(x)=20*x*exp(20-1/x), 0<x<1/20
    using rejection sampling with transformed variable Z=1/X.
    """
    samples = []
    M = 1.05
    
    while len(samples) < n_samples:
        # Proposal distribution: Z = 20 + Exp(1), so Z >= 20
        Z = 20 + np.random.exponential(scale=1.0)
        
        # Acceptance probability
        accept_prob = 20 * (1 / Z + 1 / Z**2) / M
        
        U = np.random.uniform(0, 1)
        
        if U <= accept_prob:
            X = 1 / Z
            samples.append(X)
    
    return np.array(samples)

# Small test
problem1_samples_2 = problem1_inversion_2(10000)
problem1_samples_2[:5], problem1_samples_2.shape

## Problem 1 Local Format Test

Eta original exam-er local test-er moto. Eita run kore dekho variables correct format-e ache kina.

In [ ]:
# Problem 1 format test

try:
    assert isinstance(problem1_inversion(10), np.ndarray)
    print("Good: problem1_inversion returns numpy array")
except Exception:
    print("Try again: problem1_inversion should return numpy array")

try:
    assert isinstance(problem1_samples, np.ndarray)
    print("Good: problem1_samples is numpy array")
except Exception:
    print("Try again: problem1_samples should be numpy array")

try:
    assert isinstance(problem1_integral, float)
    print("Good: problem1_integral is float")
except Exception:
    print("Try again: problem1_integral should be float")

try:
    assert isinstance(problem1_interval, (list, tuple))
    assert len(problem1_interval) == 2
    print("Good: problem1_interval is tuple/list of length 2")
except Exception as e:
    print("Try again:", e)

try:
    assert isinstance(problem1_inversion_2(10), np.ndarray)
    print("Good: problem1_inversion_2 returns numpy array")
except Exception:
    print("Try again: problem1_inversion_2 should return numpy array")

# Problem 2 — Spam Logistic Model and Calibration

Features:

- \(X_1=1\) if word `free` appears, otherwise 0
- \(X_2=1\) if word `prize` appears, otherwise 0
- \(X_3=1\) if word `win` appears, otherwise 0

Label:

- spam = 1
- ham/not spam = 0

Split:

- train = 40%
- calibration = 20%
- test = 40%

In [ ]:
# Problem 2, Part 1

from sklearn.model_selection import train_test_split

# Make sure data/spam.csv exists in your working directory.
df = pd.read_csv("data/spam.csv")

words = ["free", "prize", "win"]

# X matrix: each row = one email, each column = whether word exists or not
problem2_X = np.array([
    [1 if word in str(text).lower() else 0 for word in words]
    for text in df["v2"]
])

# Y vector: spam = 1, ham = 0
problem2_Y = np.array([
    1 if label == "spam" else 0
    for label in df["v1"]
])

# First split: 40% train, 60% temporary
problem2_X_train, X_temp, problem2_Y_train, Y_temp = train_test_split(
    problem2_X,
    problem2_Y,
    train_size=0.4,
    random_state=0
)

# From remaining 60%, we need 20% calibration and 40% test.
# So inside temporary data, calibration fraction = 20/60 = 1/3.
problem2_X_calib, problem2_X_test, problem2_Y_calib, problem2_Y_test = train_test_split(
    X_temp,
    Y_temp,
    train_size=1/3,
    random_state=0
)

print(
    problem2_X_train.shape,
    problem2_X_calib.shape,
    problem2_X_test.shape,
    problem2_Y_train.shape,
    problem2_Y_calib.shape,
    problem2_Y_test.shape
)

## Problem 2, Part 2 — Logistic Regression Loss

Model:

$$
P(Y=1|X)=G(\beta_0+\beta X)
$$

where logistic function:

$$
G(z)=\frac{e^z}{1+e^z}
$$

Loss function:

$$
L=-\frac{1}{n}\sum_{i=1}^{n}\left[y_i\log(p_i)+(1-y_i)\log(1-p_i)\right]
$$

Simple meaning:

- Actual spam hole model-er probability high howa uchit.
- Actual ham hole model-er probability low howa uchit.
- Wrong confidence dile loss bere jay.

In [ ]:
# Problem 2, Part 2

class ProportionalSpam(object):
    def __init__(self):
        self.coeffs = None
        self.result = None
    
    def loss(self, X, Y, coeffs):
        """
        Negative log-likelihood loss for logistic regression.
        coeffs[0] = beta0/intercept
        coeffs[1:] = beta coefficients
        """
        beta0 = coeffs[0]
        beta = coeffs[1:]
        
        z = np.dot(X, beta) + beta0
        p = np.exp(z) / (1 + np.exp(z))
        
        # Numerical safety: avoid log(0)
        eps = 1e-15
        p = np.clip(p, eps, 1 - eps)
        
        loss_value = -np.mean(Y * np.log(p) + (1 - Y) * np.log(1 - p))
        return loss_value

    def fit(self, X, Y):
        from scipy import optimize

        opt_loss = lambda coeffs: self.loss(X, Y, coeffs)
        initial_arguments = np.zeros(shape=X.shape[1] + 1)
        self.result = optimize.minimize(opt_loss, initial_arguments, method='cg')
        self.coeffs = self.result.x
    
    def predict(self, X):
        if self.coeffs is not None:
            G = lambda x: np.exp(x) / (1 + np.exp(x))
            return np.round(
                10 * G(np.dot(X, self.coeffs[1:]) + self.coeffs[0])
            ) / 10

## Loss Function Test

Original exam test value should match:

$$
1.2828629432232497
$$

In [ ]:
# Problem 2 loss test

try:
    test_instance = ProportionalSpam()
    test_loss = test_instance.loss(
        np.array([[1, 0, 1], [0, 1, 1]]),
        np.array([1, 0]),
        np.array([1.2, 0.4, 0.3, 0.9])
    )
    print("test_loss =", test_loss)
    assert np.abs(test_loss - 1.2828629432232497) < 1e-6
    print("Good: loss is correct for the test point")
except Exception:
    print("Try again: loss is not correct")

## Problem 2, Part 3 — Train Model and Calibration Model

First logistic model train korbo train data diye.

Then calibration data-te model probability predict korbe.

Calibration model shikhbe:

> Model jokhon 0.8 bole, actual spam probability roughly koto?

Here calibrator = `DecisionTreeRegressor`.

In [ ]:
# Problem 2, Part 3

from sklearn.tree import DecisionTreeRegressor

problem2_ps = ProportionalSpam()
problem2_ps.fit(problem2_X_train, problem2_Y_train)

# Predictions on calibration set, shape must be (n_samples, 1)
problem2_X_pred = problem2_ps.predict(problem2_X_calib).reshape(-1, 1)

# Train calibrator
problem2_calibrator = DecisionTreeRegressor(random_state=0)
problem2_calibrator.fit(problem2_X_pred, problem2_Y_calib)

print("Logistic coefficients:", problem2_ps.coeffs)
print("Calibration prediction shape:", problem2_X_pred.shape)

## Problem 2, Part 4 — Test Prediction, 0-1 Loss, and 99% CI

Final steps:

1. Test data-te logistic probability predict.
2. Calibrator diye calibrated probability ber kori.
3. Probability \(\ge 0.5\) hole spam, otherwise ham.
4. 0-1 loss = wrong predictions / total predictions.

99% Hoeffding interval:

$$
\epsilon=\sqrt{\frac{\log(2/\alpha)}{2n}}
$$

Here:

$$
\alpha=0.01
$$

In [ ]:
# Problem 2, Part 4

# Logistic model probabilities on test data
problem2_test_raw_predictions = problem2_ps.predict(problem2_X_test).reshape(-1, 1)

# Calibrated probabilities
problem2_final_predictions = problem2_calibrator.predict(problem2_test_raw_predictions)

# Bayes classifier: probability >= 0.5 means spam
problem2_final_labels = (problem2_final_predictions >= 0.5).astype(int)

# 0-1 loss
problem2_01_loss = float(np.mean(problem2_final_labels != problem2_Y_test))

# 99% confidence interval using Hoeffding
n = len(problem2_Y_test)
alpha = 0.01
epsilon = np.sqrt(np.log(2 / alpha) / (2 * n))

problem2_interval = (
    float(problem2_01_loss - epsilon),
    float(problem2_01_loss + epsilon)
)

print("0-1 loss:", problem2_01_loss)
print("99% confidence interval:", problem2_interval)

# Problem 3 — Markov Chain Helper

Ei part exact solve korte **MarkovA.png, MarkovB.png, MarkovC.png, MarkovD.png** diagram lagbe.

Diagram chara transition probabilities jana jay na, tai exact matrix fill kora possible na.

But niche helper functions diye dilam. Tumi matrix fill korlei code automatically:

- irreducible kina
- periods
- aperiodic kina
- stationary distribution
- reversible kina

ber kore dibe.

In [ ]:
# Problem 3 helper functions

from math import gcd
from functools import reduce

def reachable_matrix(P):
    """Return reachability matrix: R[i,j]=True if j reachable from i."""
    P = np.array(P, dtype=float)
    n = P.shape[0]
    A = P > 0
    R = A.copy()
    power = A.copy()
    for _ in range(n - 1):
        power = power @ A
        R = R | (power > 0)
    return R

def is_irreducible(P):
    R = reachable_matrix(P)
    return bool(np.all(R))

def state_periods(P, max_steps=100):
    """
    Estimate exact periods for finite small chains by checking return times up to max_steps.
    For exam-size chains this is usually enough.
    """
    P = np.array(P, dtype=float)
    n = P.shape[0]
    periods = []
    P_power = np.eye(n)
    return_times = [[] for _ in range(n)]
    
    for step in range(1, max_steps + 1):
        P_power = P_power @ P
        for i in range(n):
            if P_power[i, i] > 1e-12:
                return_times[i].append(step)
    
    for times in return_times:
        if len(times) == 0:
            periods.append(0)  # no return found
        else:
            periods.append(reduce(gcd, times))
    
    return np.array(periods)

def is_aperiodic(P):
    periods = state_periods(P)
    return bool(np.all(periods == 1))

def stationary_distribution(P):
    """
    Solve pi P = pi and sum(pi)=1.
    Returns one stationary distribution for finite chain.
    """
    P = np.array(P, dtype=float)
    n = P.shape[0]
    A = P.T - np.eye(n)
    A[-1, :] = np.ones(n)
    b = np.zeros(n)
    b[-1] = 1
    try:
        pi = np.linalg.solve(A, b)
        pi[np.abs(pi) < 1e-12] = 0
        return pi
    except np.linalg.LinAlgError:
        # fallback using least squares
        pi, *_ = np.linalg.lstsq(A, b, rcond=None)
        pi[np.abs(pi) < 1e-12] = 0
        return pi

def is_reversible(P, pi=None):
    P = np.array(P, dtype=float)
    if pi is None:
        pi = stationary_distribution(P)
    n = P.shape[0]
    for i in range(n):
        for j in range(n):
            if not np.isclose(pi[i] * P[i, j], pi[j] * P[j, i], atol=1e-8):
                return False
    return True

def analyze_markov_chain(P, name="Chain"):
    P = np.array(P, dtype=float)
    print(f"--- {name} ---")
    print("Transition matrix:")
    print(P)
    print("Row sums:", P.sum(axis=1))
    print("Irreducible:", is_irreducible(P))
    periods = state_periods(P)
    print("Periods:", periods)
    print("Aperiodic:", np.all(periods == 1))
    pi = stationary_distribution(P)
    print("Stationary distribution:", pi)
    print("Check pi P:", pi @ P)
    print("Reversible:", is_reversible(P, pi))

## Problem 3 Matrix Template

Nicher `None` gula replace kore actual transition matrices boshabe.

Example matrix format:

```python
problem3_A = np.array([
    [0.0, 0.5, 0.5],
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0]
])
```

Important:

- row = current state
- column = next state
- each row sum must be 1

In [ ]:
# Problem 3, Part 1 — Fill these using the diagrams

problem3_A = None  # replace with np.array([...])
problem3_B = None  # replace with np.array([...])
problem3_C = None  # replace with np.array([...])
problem3_D = None  # replace with np.array([...])

# Example use after filling:
# analyze_markov_chain(problem3_A, "Markov chain A")

In [ ]:
# Problem 3 auto-answer template
# Run this cell after filling problem3_A, problem3_B, problem3_C, problem3_D.

chains = {
    "A": problem3_A,
    "B": problem3_B,
    "C": problem3_C,
    "D": problem3_D,
}

for name, P in chains.items():
    if P is None:
        print(f"Markov chain {name}: matrix not filled yet.")
    else:
        analyze_markov_chain(P, f"Markov chain {name}")
        print()

# Final Notes

For exam answer writing:

- **Transition matrix**: diagram theke row-wise probabilities likhba.
- **Irreducible**: every state theke every state-e jawa jay kina check korba.
- **Aperiodic**: return times-er gcd 1 kina check korba.
- **Stationary distribution**: solve \(\pi P=\pi\), \(\sum \pi_i=1\).
- **Reversible**: check \(\pi_iP_{ij}=\pi_jP_{ji}\).